# 02 · Revisión de todas las fuentes

**Objetivo:** documentar qué contiene cada fuente: rango de fechas, bandas, resolución, rol en el proyecto y limitaciones.

**Solo lee datos.** No escribe archivos. Las consultas a GEE son de metadatos.

**Estado:** notebook nuevo, todavía sin ejecutar.

**Orden del proyecto:** 01 límite → **02 fuentes** → 03 viabilidad de años → 04 catálogo de indicadores → 05 visor de imágenes → 06 descarga.


## 1. Configuración


In [ ]:
from datetime import date, datetime
from pathlib import Path
import hashlib
import io
import json
import re

import ee
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

GEE_PROJECT = "viirs-peru"
ANIO_INI, ANIO_FIN = 2010, 2025  # 15 periodos anuales: 2010 -> 2025

# Bandas por sensor: nombre común -> banda original
BANDAS_LANDSAT = {
    "L5": {"blue": "SR_B1", "green": "SR_B2", "red": "SR_B3", "nir": "SR_B4", "swir1": "SR_B5", "swir2": "SR_B7"},
    "L7": {"blue": "SR_B1", "green": "SR_B2", "red": "SR_B3", "nir": "SR_B4", "swir1": "SR_B5", "swir2": "SR_B7"},
    "L8": {"blue": "SR_B2", "green": "SR_B3", "red": "SR_B4", "nir": "SR_B5", "swir1": "SR_B6", "swir2": "SR_B7"},
    "L9": {"blue": "SR_B2", "green": "SR_B3", "red": "SR_B4", "nir": "SR_B5", "swir1": "SR_B6", "swir2": "SR_B7"},
}
BANDAS_S2 = {"blue": "B2", "green": "B3", "red": "B4", "nir": "B8", "swir1": "B11", "swir2": "B12"}
COLECCIONES_LANDSAT = {
    "L5": "LANDSAT/LT05/C02/T1_L2",
    "L7": "LANDSAT/LE07/C02/T1_L2",
    "L8": "LANDSAT/LC08/C02/T1_L2",
    "L9": "LANDSAT/LC09/C02/T1_L2",
}
COLECCION_S2 = "COPERNICUS/S2_SR_HARMONIZED"
MB_ASSET = "projects/mapbiomas-public/assets/peru/collection4/mapbiomas_peru_collection4_coverage_v1"


def buscar_raiz(inicio=None):
    """Sube por las carpetas hasta encontrar la raíz del proyecto (la que tiene el límite provincial).
    Se usa el GeoPackage como marcador porque puede existir una carpeta data/raw dentro de notebooks/."""
    inicio = Path(inicio or Path.cwd()).resolve()
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg").is_file():
            return carpeta
    raise FileNotFoundError("No encuentro data/raw/Provincia/PROVINCIA.gpkg desde esta ubicación.")


def iniciar_gee(proyecto=GEE_PROJECT):
    try:
        ee.Initialize(project=proyecto)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=proyecto)


def rango_fechas(col):
    """Primera y última fecha de una colección, como texto."""
    d = ee.Dictionary({
        "min": col.aggregate_min("system:time_start"),
        "max": col.aggregate_max("system:time_start"),
    }).getInfo()
    if d["min"] is None:
        return "sin fecha"
    a = pd.to_datetime(d["min"], unit="ms").strftime("%Y-%m-%d")
    b = pd.to_datetime(d["max"], unit="ms").strftime("%Y-%m-%d")
    return f"{a} → {b}"


def md5(ruta):
    """Suma MD5 de un archivo, leído por bloques."""
    h = hashlib.md5()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


iniciar_gee()
RAIZ = buscar_raiz()
GPKG = RAIZ / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg"
gdf = gpd.read_file(GPKG)
prov = gdf[(gdf["ccdd"] == "08") & (gdf["ccpp"] == "01")]
assert len(prov) == 1, "El filtro de provincia no devolvió exactamente 1 registro"
aoi = ee.Geometry(prov.geometry.iloc[0].simplify(0.0001, preserve_topology=True).__geo_interface__)
print("Provincia:", prov["nombprov"].iloc[0], "| ventana:", ANIO_INI, "-", ANIO_FIN)


## 2. Límite provincial (INEI)

El límite viene de **INEI, límites provinciales** (`data/raw/Provincia/PROVINCIA.gpkg`). La columna `fuente` del archivo dice "V Censo Nacional Económico"; ese dato se documenta tal cual.


In [ ]:
FILAS = []  # cada sección agrega una fila a la tabla final

print("CRS:", gdf.crs)
print("Columna fuente:", prov["fuente"].iloc[0])
print("Área (km², EPSG:32719):", round(prov.to_crs(epsg=32719).area.iloc[0] / 1e6, 1))
print("Límites (lon/lat):", [round(v, 4) for v in prov.total_bounds])


## 3. Sentinel-2 SR Harmonized


In [ ]:
col = ee.ImageCollection(COLECCION_S2).filterBounds(aoi)
img = ee.Image(col.first())
FILAS.append({
    "fuente": "Sentinel-2 SR Harmonized",
    "producto": "reflectancia superficial",
    "rango_fechas": rango_fechas(col),
    "escala_m": img.projection().nominalScale().getInfo(),
    "bandas": ", ".join(img.bandNames().getInfo()),
    "rol": "complemento (2019–2025)",
})
print("Bandas que usaremos (nombre común -> banda):", BANDAS_S2)


## 4. Landsat C2 L2 por sensor

Los cuatro sensores se estandarizan con la misma escala y los mismos nombres de banda. Cada sensor tiene números de banda distintos.


In [ ]:
for sensor, cid in COLECCIONES_LANDSAT.items():
    col = ee.ImageCollection(cid).filterBounds(aoi)
    img = ee.Image(col.first())
    FILAS.append({
        "fuente": f"Landsat {sensor[1]} C2 L2",
        "producto": "reflectancia superficial",
        "rango_fechas": rango_fechas(col),
        "escala_m": img.projection().nominalScale().getInfo(),
        "bandas": ", ".join(img.bandNames().getInfo()),
        "rol": "serie principal",
    })

print("Bandas por sensor (nombre común -> banda original):")
print(pd.DataFrame(BANDAS_LANDSAT).T)


## 5. ESA WorldCover (2020 y 2021)


In [ ]:
for etiqueta, cid in [("WorldCover v100 (2020)", "ESA/WorldCover/v100"),
                      ("WorldCover v200 (2021)", "ESA/WorldCover/v200")]:
    col = ee.ImageCollection(cid).filterBounds(aoi)
    img = ee.Image(col.first())
    FILAS.append({
        "fuente": etiqueta,
        "producto": "clases de cobertura",
        "rango_fechas": rango_fechas(col),
        "escala_m": img.projection().nominalScale().getInfo(),
        "bandas": ", ".join(img.bandNames().getInfo()),
        "rol": "referencia puntual",
    })


## 6. SRTM (elevación)


In [ ]:
img = ee.Image("USGS/SRTMGL1_003")
FILAS.append({
    "fuente": "SRTM",
    "producto": "modelo de elevación",
    "rango_fechas": "sin fecha (modelo fijo)",
    "escala_m": img.projection().nominalScale().getInfo(),
    "bandas": ", ".join(img.bandNames().getInfo()),
    "rol": "factor (altitud)",
})


## 7. MapBiomas Perú Colección 4


In [ ]:
mb = ee.Image(MB_ASSET)
bandas_mb = mb.bandNames().getInfo()
anios_mb = sorted(int(b.split("_")[1]) for b in bandas_mb if b.startswith("classification_"))
FILAS.append({
    "fuente": "MapBiomas Perú C4",
    "producto": "cobertura y uso del suelo",
    "rango_fechas": f"{anios_mb[0]} a {anios_mb[-1]} (una banda por año)",
    "escala_m": mb.projection().nominalScale().getInfo(),
    "bandas": f"{len(bandas_mb)} bandas classification_YYYY",
    "rol": "referencia principal",
})


## 8. GEOBOSQUES (archivos locales)

Se revisa lo que ya tenemos en `data/raw/GEOBOSQUES/`: la estructura del JSON, los años de las imágenes y si hay imágenes repetidas.


In [ ]:
def describir(obj, prefijo="", nivel=0, max_nivel=3):
    """Imprime la estructura de un JSON: claves, tipos y un valor corto."""
    if nivel > max_nivel:
        return
    if isinstance(obj, dict):
        for k, v in obj.items():
            vista = "" if isinstance(v, (dict, list)) else f" = {str(v)[:80]}"
            print(f"{prefijo}{k} ({type(v).__name__}){vista}")
            describir(v, prefijo + "  ", nivel + 1, max_nivel)
    elif isinstance(obj, list):
        print(f"{prefijo}[lista de {len(obj)}]")
        if obj:
            describir(obj[0], prefijo + "  ", nivel + 1, max_nivel)


carpeta_gb = RAIZ / "data" / "raw" / "GEOBOSQUES"
ruta_json = carpeta_gb / "info_bosque_humedo_2025.json"
if ruta_json.exists():
    info_gb = json.loads(ruta_json.read_text(encoding="utf-8"))
    print("Estructura de info_bosque_humedo_2025.json:")
    describir(info_gb)
else:
    print("No encuentro", ruta_json)


In [ ]:
pngs = sorted((carpeta_gb / "anual").glob("bosque_*_provincia_cusco.png"))
anios_png = sorted(int(re.search(r"bosque_(\d{4})_", p.name).group(1)) for p in pngs)
faltan = [a for a in range(2016, 2026) if a not in anios_png]
print("Años con imagen:", anios_png)
print("Años faltantes entre 2016 y 2025:", faltan or "ninguno")

verificacion = pd.DataFrame({
    "archivo": [p.name for p in pngs],
    "bytes": [p.stat().st_size for p in pngs],
    "md5": [md5(p) for p in pngs],
})
print(verificacion)
print("Imágenes con MD5 repetido:", int(verificacion["md5"].duplicated(keep=False).sum()))

FILAS.append({
    "fuente": "GEOBOSQUES (imágenes WMS)",
    "producto": "bosque y pérdida de bosque (visual)",
    "rango_fechas": f"imágenes locales: {anios_png[0]} a {anios_png[-1]}" if anios_png else "sin imágenes",
    "escala_m": "no aplica (PNG)",
    "bandas": "no aplica",
    "rol": "referencia (cobertura parcial)",
})


In [ ]:
import xml.etree.ElementTree as ET

WMS_URL = "https://gis.bosques.gob.pe/server/services/Interoperabilidad/bosque_humedo_2025/MapServer/WMSServer"
resp = requests.get(WMS_URL, params={"service": "WMS", "request": "GetCapabilities"}, timeout=120)
resp.raise_for_status()
raiz_xml = ET.fromstring(resp.content)


def etiqueta_xml(el):
    return el.tag.split("}")[-1]


capas_wms = []
for el in raiz_xml.iter():
    if etiqueta_xml(el) == "Layer":
        nombre = next((c.text for c in el if etiqueta_xml(c) == "Name"), None)
        titulo = next((c.text for c in el if etiqueta_xml(c) == "Title"), None)
        if nombre:
            capas_wms.append({"Name": nombre, "Title": titulo})

capas_wms = pd.DataFrame(capas_wms)
print("Capas del WMS (elige a mano la de bosque húmedo 2025):")
print(capas_wms)

# No se elige automáticamente. Escribe aquí el Name correcto de la tabla de arriba.
CAPA_WMS = None


In [ ]:
if CAPA_WMS:
    import geemap
    from ipyleaflet import WMSLayer

    mapa_gb = geemap.Map()
    mapa_gb.centerObject(aoi, 9)
    mapa_gb.add_layer(WMSLayer(url=WMS_URL, layers=CAPA_WMS, format="image/png",
                               transparent=True, name="GeoBosques 2025"))
    mapa_gb.addLayer(ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(aoi)]), 1, 2),
                     {"palette": "000000"}, "Límite provincia")
    mapa_gb
else:
    print("CAPA_WMS está vacía. Asígnale el Name correcto y vuelve a ejecutar esta celda.")


## 9. OpenStreetMap (red vial)

Es una foto del 2026-10-08. No tiene historial por año.


In [ ]:
import pyogrio

OSM_GPKG = RAIZ / "data" / "raw" / "OpenStreetMap" / "peru.gpkg"
readme = RAIZ / "data" / "raw" / "OpenStreetMap" / "README"
if readme.exists():
    print(readme.read_text(encoding="utf-8", errors="replace")[:600])

capas_osm = [c[0] for c in pyogrio.list_layers(OSM_GPKG)]
print("Capas en peru.gpkg:", len(capas_osm))

info_vias = pyogrio.read_info(OSM_GPKG, layer="gis_osm_roads_free")
print("Tramos de vía (registros):", info_vias["features"])
print("Campos:", list(info_vias["fields"]))

FILAS.append({
    "fuente": "OpenStreetMap (red vial)",
    "producto": "red vial",
    "rango_fechas": "foto 2026-10-08, sin historial",
    "escala_m": "vectorial",
    "bandas": "no aplica",
    "rol": "factor (distancia a vías), sin historial",
})


## 10. Tabla resumen de fuentes

Revisa la columna `rol` y las limitaciones antes de pasar al notebook 03.


In [ ]:
revision = pd.DataFrame(FILAS)
revision
